# 第16周 · 作业：欧拉步数的定量评估 · MNIST 上的 Flow Matching · Capstone 基线（预计约6小时）

| 部分 | 内容 | 建议时间 |
| --- | --- | --- |
| A | 研究实验：欧拉采样步数 {2, 5, 20, 100} × 3 个种子，能量距离 + k-NN precision；结果表、误差棒图、实验记录 | 2.5 小时 |
| B | MNIST 无条件 Flow Matching（工具包的 `SmallUNet`，14×14）+ 一个改进假设 | 1.5 小时 |
| C | Capstone 基线结果写入实验记录本 | 1.5 小时 |
| D | 代码讲解 | 0.5 小时 |

**规则**：可以使用 AI 编程助手写代码，但必须完成 D 部分的"代码讲解"（用自己的话解释关键代码，并注明哪些部分由 AI 生成），且可能被随机抽查 5 分钟口头讲解。

**评分**：A 按研究实验评分表（设计 30% · 执行 30% · 分析与解释 30% · 表达 10%）；B 看流程是否正确、改进假设是否可检验；C 对照讲义 S4 的展示内容清单。

**运行时间**：A 在 CPU 上约半分钟。B 在 CPU 上约 3 分钟（Colab GPU 约半分钟）——为了在 CPU 上也能跑完，B 用 14×14 的 MNIST、1500 步；有 GPU 的同学可以按 B 部分末尾的提示放大。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torchvision.utils import make_grid
from zero2researcher import data, experiments, generative
from zero2researcher.checks import week16 as ck

# A 部分（2D 小网络）固定用 CPU；B 部分（MNIST U-Net）自动选择：Colab 上是 CUDA，Apple 芯片是 MPS，否则 CPU
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| B 部分的 device:", device)

## A · 研究实验：欧拉采样步数与样本质量

**研究问题**：同一个训练好的 Flow Matching 模型，欧拉采样步数如何影响 2D 样本的质量？两个指标（能量距离与 k-NN precision）给出的结论一致吗？

实验规格：

- **自变量**：欧拉采样步数 $n \in \{2, 5, 20, 100\}$；
- **数据**：工具包的月牙 `generative.sample_moons` 与双高斯 `generative.sample_gaussian_mixture`（默认参数）；
- **固定**：速度场网络 3→128→128→2（SiLU），Adam 1e-3，batch 512，训练 3000 步；**每个 (数据集, 种子) 只训练一个模型**，四种步数都用它采样——这样步数是唯一改变的东西；
- **种子** 0、1、2：种子同时决定初始化、训练数据流和采样噪声；
- **评估**：每次生成 2000 个点，与一份**固定的**真实参考样本（2000 个点，`rng=2026`）比较，报告能量距离与 k-NN precision（$k = 5$）的均值 ± 标准差；先量"真 vs 真"的噪声地板。

**A0 · 预注册**（跑任何代码之前写下，跑完不许改）：你预期两个指标随步数怎样变化？到多少步之后再加步数就"不值得"了？两个指标的结论会一致吗？什么样的结果会推翻你的预期？

- 预期：
- 什么结果会推翻它：

下面两个函数与讲义完全相同，原样给出：

In [ ]:
class VelocityMLP(nn.Module):
    """速度场 u_θ(x, t)：输入 2D 位置 x 与时间 t，输出 2D 速度（讲义 S2 原样）"""

    def __init__(self, hidden=128, depth=2):
        super().__init__()
        layers, d = [], 3
        for _ in range(depth):
            layers += [nn.Linear(d, hidden), nn.SiLU()]
            d = hidden
        layers.append(nn.Linear(d, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, x, t):
        t = torch.as_tensor(t, dtype=x.dtype).reshape(-1, 1).expand(len(x), 1)
        return self.net(torch.cat([x, t], dim=1))


def euler_integrate(u, x0, n_steps, t_start=0.0, t_end=1.0):
    """欧拉法（讲义 S1 原样）：返回整条轨迹 (n_steps + 1, *x0.shape)"""
    h = (t_end - t_start) / n_steps
    x, path = x0, [x0]
    for i in range(n_steps):
        x = x + h * u(x, t_start + i * h)
        path.append(x)
    return torch.stack(path)


DATA = {"moons": generative.sample_moons, "gaussian mixture": generative.sample_gaussian_mixture}
REFS = {name: sample(2000, rng=2026) for name, sample in DATA.items()}      # 固定的真实参考样本
for name, sample in DATA.items():
    eds = [generative.energy_distance(sample(2000, rng=s), REFS[name]) for s in range(10)]
    precs = [generative.knn_precision(sample(2000, rng=s), REFS[name]) for s in range(10)]
    print(f"{name:17s} 噪声地板（10 次真 vs 真）：能量距离 {np.mean(eds):+.4f} ± {np.std(eds, ddof=1):.4f}"
          f" | precision {np.mean(precs):.3f} ± {np.std(precs, ddof=1):.3f}")

**A1** · 写 `fm_loss(model, x1)`（讲义 S2）与 `train_fm(sample_data, seed, steps=3000)`：固定种子 → 新建 `VelocityMLP()` → Adam(1e-3) 训练 `steps` 步，每步 `sample_data(512)` 抽一批新数据 → 返回训练好的模型。**种子要在建模型之前设置**。

In [ ]:
def fm_loss(model, x1):
    # 在这里写你的代码
    raise NotImplementedError


def train_fm(sample_data, seed, steps=3000):
    # 在这里写你的代码
    raise NotImplementedError


ck.fm_loss(fm_loss)

**A2** · 写 `fm_sample(model, n, n_steps, seed)`：用 `torch.Generator().manual_seed(seed)` 抽 `n` 个二维标准正态噪声，在 `torch.no_grad()` 下用 `euler_integrate` 从 $t = 0$ 走到 $t = 1$，**只返回终点** `(n, 2)`。

In [ ]:
def fm_sample(model, n, n_steps, seed):
    # 在这里写你的代码
    raise NotImplementedError


smoke = fm_sample(VelocityMLP(), 5, 3, seed=0)
print("冒烟测试：未训练模型采样的形状", tuple(smoke.shape))

**A3** · 主循环：对每个数据集、每个种子训练**一个**模型，再用四种步数各采样 2000 个点（采样种子用同一个 `seed`），与 `REFS[数据集]` 比较。要求的结构（自查会检查）：

```python
results[数据集名][步数] = {"ed": [3 个种子的能量距离], "precision": [3 个种子的 precision]}
```

In [ ]:
STEPS, SEEDS = (2, 5, 20, 100), (0, 1, 2)
results = {name: {n: {"ed": [], "precision": []} for n in STEPS} for name in DATA}
t0 = time.time()
# 在这里写你的代码
raise NotImplementedError
print(f"{len(DATA) * len(SEEDS)} 个模型的训练与评估共用时 {time.time() - t0:.1f} 秒")
ck.steps_trend(results)

**A4** · 结果表 `table`（pandas DataFrame）：行 = 步数；列 = 每个数据集的"能量距离"与"precision"，格子里写 `均值 ± 标准差`（用 `experiments.mean_std`）。

In [ ]:
# 在这里写你的代码
raise NotImplementedError
table

**A5** · 误差棒图：两个子图（能量距离、precision），横轴步数（对数刻度），每个数据集一条线，误差棒 = 标准差。能量距离跨了好几个数量级、又可能在 0 附近为负，纵轴用 `ax.set_yscale("symlog", linthresh=1e-3)`。标题写结论。

In [ ]:
# 在这里写你的代码
raise NotImplementedError


**A6 · 分析问题**（每题两三句话，引用你表里的数字）

1. 从 2 步到 100 步，两个指标各改善了多少？从哪一步开始，再增加步数带来的改善**小于种子间的标准差**（用第 6 周的三分法下结论）？
2. 2 步时种子间的标准差与 100 步时相比如何？为什么"至少 3 个种子"在这里必不可少？
3. 两个指标在所有步数上给出的排序一致吗？结合讲义 S3 的棋盘格 / 螺旋：如果只报告能量距离，什么情况下会得出错误结论？
4. 能不能拿月牙的能量距离和双高斯的能量距离比较、说"模型在哪份数据上学得更好"？为什么？（提示：数据尺度、两个模式的比例对能量距离的影响）
5. 采样 100 步比 20 步慢 5 倍。你会推荐用几步？这个推荐依赖哪些**固定超参数**（网络大小、训练步数、数据集）——换一组固定超参数，结论可能怎样变？

（在这里回答）

**A7 · 实验记录**（正式提交格式，"结果"一栏写均值 ± 标准差并引用上面的表和图；"局限"一栏讨论固定超参数对结论的限定）

## 实验记录 #__ · 日期：____

- 研究问题：
- 假设（预期结果，以及什么结果会推翻它）：
- 自变量（这次只改变什么）：
- 控制变量（哪些保持不变；哪些干扰变量单独调优）：
- 基线：
- 数据与划分方式（如何避免泄漏）：
- 随机种子：
- 结果（数值：均值±标准差；图：链接或插图）：
- 解释：
- 局限：
- 下一步：

## B · MNIST 上的无条件 Flow Matching

同样的算法搬到图像上。图像的速度场网络换成工具包的 `generative.SmallUNet`——课程不教 U-Net 的内部结构，你只需要它的接口：`model(x, t)` 输入 `(B, 1, 14, 14)` 的图像与每张图一个时间，输出同形状的速度。

数据用工具包的 `generative.load_mnist`：缩放到 $[-1, 1]$（背景 −1、笔画 +1），并平均池化到 **14×14**——第六阶段调研实测，28×28 在 CPU 上几分钟内训不出可辨认的数字，14×14 是现实的上限。训练子集取 10000 张。

In [ ]:
x_train, y_train = generative.load_mnist(size=14, train=True, n=10000, seed=0)
print("训练图像", tuple(x_train.shape), "| 像素范围", float(x_train.min()), "到", float(x_train.max()))


def show_grid(images, nrow=8, title=""):
    grid = make_grid(images[:nrow * nrow].detach().cpu().clamp(-1, 1), nrow=nrow, normalize=True, value_range=(-1, 1))
    plt.figure(figsize=(4.5, 4.5))
    plt.imshow(grid.permute(1, 2, 0))
    plt.axis("off")
    plt.title(title)
    plt.show()


show_grid(x_train, title="Real MNIST digits at 14x14")

**B1** · 写 `fm_loss_images(model, x1)`。它和 A1 的 `fm_loss` **只差一处**：图像是 `(B, 1, 14, 14)`，所以时间要抽成 `(B, 1, 1, 1)` 才能广播（`device=x1.device`，让它和图像在同一个设备上）。

In [ ]:
def fm_loss_images(model, x1):
    # 在这里写你的代码
    raise NotImplementedError


**B2** · 训练循环：每步从 `x_train` 里随机抽 128 张（`torch.randint`）、搬到 `device`、算 `fm_loss_images`、五步核心更新。优化器 AdamW、学习率 2e-3（调研实测的设置），1500 步。

In [ ]:
experiments.set_seed(0)
model = generative.SmallUNet(img_size=14).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3)
print("SmallUNet 参数量:", generative.count_parameters(model))
STEPS_MNIST, BATCH = 1500, 128
losses = []
t0 = time.time()
model.train()
for step in range(STEPS_MNIST):
    # 在这里写你的代码
    raise NotImplementedError
    losses.append(loss.item())
    if (step + 1) % 500 == 0:
        print(f"step {step + 1:5d} | 最近 100 步平均损失 {np.mean(losses[-100:]):.4f} | 已用 {time.time() - t0:.0f} 秒")
train_seconds = time.time() - t0

**B3** · 写 `fm_sample_images(model, n, n_steps=50, seed=0)`：在 CPU 上用 `torch.Generator().manual_seed(seed)` 抽 `(n, 1, 14, 14)` 的噪声、搬到 `device`，`model.eval()` 后在 `torch.no_grad()` 下用 `euler_integrate` 积分（`SmallUNet` 接受一个数作为时间），返回终点（搬回 CPU）。

In [ ]:
def fm_sample_images(model, n, n_steps=50, seed=0):
    # 在这里写你的代码
    raise NotImplementedError


samples = fm_sample_images(model, 64)
ck.mnist_samples_shape(samples)
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(np.convolve(losses, np.ones(50) / 50, mode="valid"))
ax.set_xlabel("Step (moving average of 50)")
ax.set_ylabel("FM loss")
ax.set_title("MNIST FM loss keeps creeping down at 1500 steps")
plt.show()
show_grid(samples, title=f"Flow Matching samples ({STEPS_MNIST} steps, 50 Euler steps)")

**B4 · 一个参考指标**。肉眼看"像不像数字"很主观。工具包提供一个在真实 14×14 数字上训练的小 CNN（第 10 周的结构），可以统计：分类器最大概率 > 0.9 的样本比例（"有把握是某个数字"），以及预测类别直方图的归一化熵（1 = 10 个数字都有，0 = 全被判成同一个数字）。**注意**：这个分类器从没见过"四不像"，对它们也可能给出高置信度——下周我们会专门做一个阴性对照。今天只把它当作一个粗略的参照，和真实数据并排比较：

In [ ]:
clf = generative.train_mnist_classifier(size=14)
x_real, _ = generative.load_mnist(size=14, train=False, n=1000, seed=1)
for name, imgs in (("real test digits", x_real), ("FM samples (256)", fm_sample_images(model, 256, seed=1))):
    s = generative.classifier_scores(clf, imgs)
    print(f"{name:18s} | 置信度 > 0.9 的比例 {s['frac_confident']:.3f} | 类别熵 {s['class_entropy']:.3f} | 各类计数 {s['pred_counts']}")

**B5 · 改进假设**（不要求实现，但要写到别人能照着做实验的程度）。看着你的样本网格和上面的指标，提出**一个**改进假设，例如时间编码方式、网络宽度（`base_ch`）、残差层数（`n_res`）、训练步数、学习率、采样步数……

- 假设（一句话，含方向："把 ___ 从 ___ 改成 ___，会使 ___ 变好/变差"）：
- 依据（你在样本或曲线里看到了什么）：
- 自变量与要保持不变的控制变量：
- 指标与判定标准（哪个数变化多少算支持你的假设？种子数？）：
- 预算（单次运行时间 × 次数；CPU 还是 Colab GPU）：

## C · Capstone 基线结果

按讲义 S4 的展示内容清单整理你的 Capstone 基线，写进下面的实验记录（与 `templates/lab_notebook.md` 相同），并在后面补三行。"结果"一栏必须同时给出基线的数值和一个**平凡参照**（随机猜 / 多数类 / "真 vs 真"的噪声地板之一）。

## 实验记录 #__ · 日期：____

- 研究问题：
- 假设（预期结果，以及什么结果会推翻它）：
- 自变量（这次只改变什么）：
- 控制变量（哪些保持不变；哪些干扰变量单独调优）：
- 基线：
- 数据与划分方式（如何避免泄漏）：
- 随机种子：
- 结果（数值：均值±标准差；图：链接或插图）：
- 解释：
- 局限：
- 下一步：

- 平凡参照及其数值：
- 单次运行耗时（在哪种硬件上）与主实验的总预算估计：
- 讲义 S4 一对一排错清单中**还没通过**的项，以及计划在哪天之前解决：

## D · 代码讲解（每份作业必交）

用你自己的话写一段（约 150–300 字）：

1. 逐段解释 A 部分的 `fm_loss`、`train_fm` 与主循环——尤其是：为什么**每个 (数据集, 种子) 只训练一个模型**、四种步数共用它？参考样本为什么要固定？
2. B 部分的 `fm_loss_images` 为什么要把时间抽成 `(B, 1, 1, 1)`？如果抽成 `(B,)` 会发生什么（试一试，读报错）？
3. 注明哪些代码由 AI 助手生成、你做了哪些检查或修改（例如：种子是否在建模型之前设置、采样时是否 `torch.no_grad()` 与 `model.eval()`、`knn_precision` 两个参数的顺序）。

讲不清楚的代码视同不是你写的——口头抽查就从这一段问起。

（在这里写）